In [54]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import re



In [55]:
name = 'csv/test027_2CH_SPI.csv'
num_CH = 2
CH_first = 0

In [56]:
df = pd.read_csv(name, sep=';', dtype=str)
df["tiempos"]  = df.index
df = df.reset_index(drop=True)
df.columns

Index(['Time', 'SPI (3,2,4,8,5,6)', 'tiempos'], dtype='object')

In [57]:
df = df.drop(columns=['SPI (3,2,4,8,5,6)'])
df.columns = ['MOSI', 'tiempos']


In [58]:
print(df.head(21))


    MOSI      tiempos
0   0x65  0,000040545
1   0xf9  0,000046180
2   0x30  0,000051940
3   0x61  0,000104025
4   0xe1  0,000109645
5   0x31  0,000115400
6   0x63  0,000168635
7   0xd0  0,000174250
8   0x30  0,000180000
9   0x61  0,000231985
10  0x6b  0,000237605
11  0x31  0,000243365
12  0x62  0,000296695
13  0xdb  0,000302330
14  0x30  0,000308090
15  0x5f  0,000360190
16  0xd8  0,000365830
17  0x31  0,000371595
18  0x61  0,000424900
19  0xae  0,000430515
20  0x30  0,000436265


In [59]:
df['MOSI'] = df['MOSI'].apply(lambda x: int(x, 16))

In [60]:
# Verifica cómo quedó
print(df.head(10))


   MOSI      tiempos
0   101  0,000040545
1   249  0,000046180
2    48  0,000051940
3    97  0,000104025
4   225  0,000109645
5    49  0,000115400
6    99  0,000168635
7   208  0,000174250
8    48  0,000180000
9    97  0,000231985


In [61]:
# Crea un diccionario para mapear los valores
mapeo = {
    48+CH_first: CH_first
}

# Comienza después de la última clave
ultima_clave = max(mapeo.keys())
ultimo_valor = max(mapeo.values())

for i in range(1, num_CH):
    nueva_clave = ultima_clave + i
    nuevo_valor = ultimo_valor + i
    mapeo[nueva_clave] = nuevo_valor

print(mapeo)

{48: 0, 49: 1}


In [62]:
# Aplica el mapeo y pon 'S' en el resto
df['type'] = df['MOSI'].astype(int).map(mapeo).fillna('S')
df = df.reset_index(drop=True)
print(df.head(10))


   MOSI      tiempos type
0   101  0,000040545    S
1   249  0,000046180    S
2    48  0,000051940  0.0
3    97  0,000104025    S
4   225  0,000109645    S
5    49  0,000115400  1.0
6    99  0,000168635    S
7   208  0,000174250    S
8    48  0,000180000  0.0
9    97  0,000231985    S


In [63]:
# Inicializamos idx como None
idx = None
# Recorremos los índices donde 'type' es distinto de 'S'
for i in df.index[df['type'] != 'S']:
    # Verificamos que haya al menos dos filas siguientes
    if (i + 2) < len(df):
        # Comprobamos que las dos siguientes filas tengan 'S'
        if (df.loc[i + 1, 'type'] == 'S') and (df.loc[i + 2, 'type'] == 'S'):
            idx = i
            break
    else:
        # Si no hay suficientes filas para verificar, no es un punto válido
        continue

# Si encontramos un índice válido, cortamos el dataframe
if idx is not None:
    df = df.loc[idx:].reset_index(drop=True)
else:
    # Si no se encontró un punto válido, el dataframe queda vacío o como prefieras manejarlo
    df = df.iloc[0:0].reset_index(drop=True)


In [64]:
print(df['type'][0])
print(df['type'][1])
print(df['type'][2])
print(df['type'][3])


0.0
S
S
1.0


In [65]:
# Diccionario para almacenar los arrays
resultados = {CH_first: []}
resultados_tiempos = {CH_first: []}



# Obtener el valor máximo actual de clave
ultima_clave = max(resultados.keys())

# Agregar N nuevas claves a ambos diccionarios
for i in range(1, num_CH):
    nueva_clave = ultima_clave + i
    resultados[nueva_clave] = []
    resultados_tiempos[nueva_clave] = []

# Iterar sobre el dataframe
for i, row in df.iterrows():
    tipo = row['type']
    tipo_tiempo = row['type']  
    if tipo in resultados:
        # Tomar las dos siguientes filas si existen
        sub_df = df.iloc[i+1:i+3]['MOSI']
        sub_df_times = df.iloc[i+1:i+3]['tiempos']
        # Guardar como array (puedes ajustar qué columnas guardar)
        resultados[tipo].append(sub_df.to_numpy())
        resultados_tiempos[tipo_tiempo].append(sub_df_times.to_numpy())
        




In [66]:

# Opcional: convertir las listas en arrays grandes (si quieres)
import numpy as np
for k in resultados:
    resultados[k] = np.concatenate(resultados[k])
    resultados_tiempos[k] = np.concatenate(resultados_tiempos[k])

# Ahora resultados[0], resultados[1], ... tienen los arrays deseados

In [67]:
for i in range (num_CH):
    print(len(resultados[i]))


1572
1566


In [68]:
# Creamos un nuevo diccionario con las filas pares eliminadas
resultados_filtrados = {}

for k, arr in resultados_tiempos.items():
    # Tomar los elementos en posiciones impares: 1, 3, 5, ...
    resultados_tiempos[k] = arr[1::2]


In [69]:
arr0 = resultados[0].flatten()

new_arr0 = []
new_arr1 = []
new_arr2 = []
new_arr3 = []
new_arr4 = []
for i in range(0, len(arr0)-1, 2):
    combined = arr0[i] * 256 + arr0[i+1]
    new_arr0.append(combined)
new_arr0 = np.array(new_arr0)



In [70]:
arr1 = resultados[1].flatten()
for i in range(0, len(arr1)-1, 2):
    combined = arr1[i] * 256 + arr1[i+1]
    new_arr1.append(combined)
new_arr1 = np.array(new_arr1)


In [71]:

arr2 = resultados[2].flatten()
for i in range(0, len(arr2)-1, 2):
    combined = arr2[i] * 256 + arr2[i+1]
    new_arr2.append(combined)
new_arr2 = np.array(new_arr2)

KeyError: 2

In [ ]:

arr3 = resultados[3].flatten()
for i in range(0, len(arr3)-1, 2):
    combined = arr3[i] * 256 + arr3[i+1]
    new_arr3.append(combined)
new_arr3 = np.array(new_arr3)

KeyError: 3

In [ ]:

arr4 = resultados[4].flatten()
for i in range(0, len(arr4)-1, 2):
    combined = arr4[i] * 256 + arr4[i+1]
    new_arr4.append(combined)
new_arr4 = np.array(new_arr4)

KeyError: 4

In [72]:

fig = go.Figure()


# Agregamos cada señal como un trazo
fig.add_trace(go.Scatter(y=new_arr0, mode='lines', name='Señal 0'))
fig.add_trace(go.Scatter(y=new_arr1, mode='lines', name='Señal 1'))
fig.add_trace(go.Scatter(y=new_arr2, mode='lines', name='Señal 2'))
# fig.add_trace(go.Scatter(y=new_arr3, mode='lines', name='Señal 3'))
# fig.add_trace(go.Scatter(y=new_arr4, mode='lines', name='Señal 4'))

# Opciones de layout
fig.update_layout(
    title='Señales muestreadas superpuestas',
    xaxis_title='Tiempo',
    yaxis_title='Valor',
    hovermode='x unified'
)

fig.show()


In [73]:

fig = go.Figure()


# Agregamos cada señal como un trazo
fig.add_trace(go.Scatter(x = resultados_tiempos[0], y=new_arr0, mode='lines', name='Señal 0'))

# Opciones de layout
fig.update_layout(
    title='Señales muestreadas superpuestas',
    xaxis_title='Tiempo',
    yaxis_title='Valor',
    hovermode='x unified'
)

fig.show()

In [74]:
fig = go.Figure()


# Agregamos cada señal como un trazo
fig.add_trace(go.Scatter(x = resultados_tiempos[1], y=new_arr1, mode='lines', name='Señal 1'))

# Opciones de layout
fig.update_layout(
    title='Señales muestreadas superpuestas',
    xaxis_title='Tiempo',
    yaxis_title='Valor',
    hovermode='x unified'
)

fig.show()

In [ ]:
fig = go.Figure()


# Agregamos cada señal como un trazo
fig.add_trace(go.Scatter(x = resultados_tiempos[2], y=new_arr2, mode='lines', name='Señal 2'))

# Opciones de layout
fig.update_layout(
    title='Señales muestreadas superpuestas',
    xaxis_title='Tiempo',
    yaxis_title='Valor',
    hovermode='x unified'
)

fig.show()

KeyError: 2

In [ ]:
fig = go.Figure()


# Agregamos cada señal como un trazo
fig.add_trace(go.Scatter(x = resultados_tiempos[3], y=new_arr3, mode='lines', name='Señal 3'))

# Opciones de layout
fig.update_layout(
    title='Señales muestreadas superpuestas',
    xaxis_title='Tiempo',
    yaxis_title='Valor',
    hovermode='x unified'
)

fig.show()

KeyError: 3

In [ ]:
fig = go.Figure()


# Agregamos cada señal como un trazo
fig.add_trace(go.Scatter(x = resultados_tiempos[4], y=new_arr4, mode='lines', name='Señal 4'))

# Opciones de layout
fig.update_layout(
    title='Señales muestreadas superpuestas',
    xaxis_title='Tiempo',
    yaxis_title='Valor',
    hovermode='x unified'
)

fig.show()

KeyError: 4

In [ ]:
ceros = np.zeros(100)  
# FFT
X = np.fft.fft(np.concatenate((ceros,(new_arr-32768))))

# Número de muestras
N = len(X)

# Frecuencias asociadas (eje x)
freqs = np.fft.fftfreq(N, 1/f)

# Magnitud (módulo)
magnitud = np.abs(X)

# Para mostrar solo la mitad positiva (frecuencias positivas)
idxs = freqs >= 0

plt.plot(freqs[idxs], magnitud[idxs])
plt.xlabel("Frecuencia (Hz)")
plt.ylabel("Magnitud")
plt.title("Espectro de la señal")
plt.show()

NameError: name 'new_arr' is not defined

In [ ]:

fig = go.Figure()

fig.add_trace(go.Scatter(
    x = freqs[idxs],
    y=magnitud[idxs],
    mode='lines',
    name='Valores concatenados'
))

fig.update_layout(
    title='FFT',
    xaxis_title='Frecuencia',
    yaxis_title='Magnitud',
    hovermode='x unified'
)

fig.show()